# Semana 9: ¿Diferencia real, o ruido?

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from scipy import stats
df = pd.read_csv(BASE + "bicicletas_dia.csv", parse_dates=["dteday"])

## 1. Población y muestra
Los 731 días como población; 2.000 muestras de 30 días.

In [ ]:
rng = np.random.default_rng(42)
medias = np.array([rng.choice(df["cnt"].values, 30, replace=False).mean() for _ in range(2000)])
sigma = df["cnt"].std(ddof=0)
print("Población: media", round(df["cnt"].mean()), " sigma", round(sigma))
print("Medias de 30 días: media", round(medias.mean()), " desviación", round(medias.std(ddof=1)), " sigma/raíz(30):", round(sigma / np.sqrt(30)))
plt.hist(medias, bins=30); plt.title("2.000 medias de 30 días"); plt.show()

## 2. Hipótesis
Escriba aquí H0 y H1 para cada variable, y el nivel de significancia, **antes** de correr la siguiente celda.

## 3. Prueba t de Welch

In [ ]:
lab, nolab = df[df["workingday"] == 1], df[df["workingday"] == 0]
print("Días:", len(lab), len(nolab))
for col in ["cnt", "casual", "registered"]:
    t = stats.ttest_ind(lab[col], nolab[col], equal_var=False)
    print(f"{col:11s} laboral {lab[col].mean():8.1f}  no laboral {nolab[col].mean():8.1f}  dif {lab[col].mean() - nolab[col].mean():7.1f}  t {t.statistic:6.2f}  p {t.pvalue:.3g}")

## 4. Intervalo bootstrap

In [ ]:
rng = np.random.default_rng(42)
a, b = lab["cnt"].values, nolab["cnt"].values
dif = np.array([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(5000)])
print("Intervalo bootstrap del 95 %:", np.percentile(dif, [2.5, 97.5]).round(0))
c12 = df[df["yr"] == 1]["cnt"]
print("Media 2012:", round(c12.mean()), " IC t 95 %:", np.round(stats.t.interval(0.95, len(c12) - 1, loc=c12.mean(), scale=stats.sem(c12))))

## 5. Grupos al azar

In [ ]:
rng = np.random.default_rng(42)
pvals = []
for _ in range(100):
    m = rng.permutation(len(df)) < len(df) // 2
    pvals.append(stats.ttest_ind(df["cnt"][m], df["cnt"][~m], equal_var=False).pvalue)
pvals = np.array(pvals)
print("p < 0,05:", (pvals < 0.05).sum(), "de 100;  p mínimo:", round(pvals.min(), 4))

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.